In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", None)

In [2]:
# Paths

PROJECT_ROOT = Path.cwd().parent

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

In [3]:
# Load Latent Features

latent_df = pd.read_parquet(
    DATA_PROCESSED / "latent_features.parquet"
)

print(latent_df.shape)

latent_df.head()

(93358, 4)


,latent_1,latent_2,latent_3,latent_4
0,1.111148,-2.120633,-0.003959,-2.057972
1,-0.132927,-2.316405,1.388605,-1.193602
2,0.905365,-2.913302,1.212862,-3.470234
3,-0.590161,-2.668145,1.046453,-1.992879
4,1.440837,-1.910079,0.204552,-1.695304


In [4]:
# Validation

validation = pd.Series({
    "rows": len(latent_df),
    "columns": len(latent_df.columns),
    "missing_values": latent_df.isna().sum().sum(),
    "duplicate_rows": latent_df.duplicated().sum()
})

validation

rows              93358
columns               4
missing_values        0
duplicate_rows      792
dtype: int64

In [5]:
# Feature Matrix
X = latent_df.copy()

print(X.shape)

(93358, 4)


In [6]:
# Train K-Means on Latent Features

BEST_K = 4

latent_kmeans = KMeans(
    n_clusters=BEST_K,
    random_state=42,
    n_init=10
)

latent_labels = latent_kmeans.fit_predict(X)

In [7]:
# Cluster Summary

latent_cluster_summary = (
    pd.Series(latent_labels)
    .value_counts()
    .sort_index()
    .to_frame("customer_count")
)

latent_cluster_summary["percentage"] = (
    latent_cluster_summary["customer_count"]
    / len(latent_labels)
    * 100
).round(2)

latent_cluster_summary

,customer_count,percentage
0,5464,5.85
1,63937,68.49
2,23380,25.04
3,577,0.62


In [8]:
# Business Profile

original_df = pd.read_parquet(
    DATA_PROCESSED / "modeling_features.parquet"
)

original_df["cluster"] = latent_labels

latent_business_profile = (
    original_df
    .groupby("cluster")
    .agg({
        "recency_days": "mean",
        "frequency": "mean",
        "monetary_value": "mean",
        "total_items": "mean",
        "avg_review_score": "mean",
        "unique_products": "mean",
        "unique_categories": "mean",
        "unique_sellers": "mean"
    })
    .round(2)
)

latent_business_profile.insert(
    0,
    "customer_count",
    original_df["cluster"].value_counts().sort_index().values
)

latent_business_profile

,customer_count,recency_days,frequency,monetary_value,total_items,avg_review_score,unique_products,unique_categories,unique_sellers
cluster,,,,,,,,,
0,5464,221.98,1.20,506.43,2.15,2.95,1.48,1.17,1.29
1,63937,242.40,1.00,87.72,1.03,4.72,1.01,0.99,1.00
2,23380,228.01,1.07,162.98,1.31,2.92,1.13,1.04,1.07
3,577,222.57,1.34,1793.69,3.03,3.42,1.56,1.13,1.32
